In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
import kagglehub
import pandas as pd
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

In [ ]:
import torch
import torch.nn as nn
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader
from torch.optim import Adam
from torchvision.transforms.functional import to_tensor
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Encode features and target using LabelEncoder
categorical_cols = img_path.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"Encoding column: {col}")
    le = LabelEncoder()
    img_path[col] = le.fit_transform(img_path[col])
img_path

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors

X_train = torch.tensor(X_train.X, dtype=torch.float32)
X_test  = torch.tensor(X_test.X, dtype=torch.float32)
y_train = torch.tensor(y_train.X, dtype=torch.long)
y_test  = torch.tensor(y_test.X, dtype=torch.long)


In [ ]:
# 2. Create TensorDataset objects
from torchvision.datasets import MNIST

#train_dataset = TensorDataset(X_train, y_train)
#test_dataset = TensorDataset(X_test, y_test)
train_dataset = MNIST(
    root='./datasets',     # Dataset storage path
    train=True,            # Use training data
    transform=to_tensor,   # Convert images to tensors
    download=True          # Download if not available
)
"""we started with creating a folder called datasets to store the dataset, then we loaded the training split, we then uploaded a
 ToTensor() transformation to: 1. Converts PIL Image / numpy array to PyTorch tensor
                               2. Scales pixel values from [0, 255] to [0, 1]
                               3.adding a channel dimension by converting the shape from (H, W) to (C, H, W)
lastly we Download the dataset if it doesn't exist in the ./datasets folder
"""
# Testing dataset
test_dataset = MNIST(
    root='./datasets',     # Dataset storage path
    train=False,           # Use test data
    transform=to_tensor,   # Convert images to tensors
    download=True          # Download if not available
)

''' same with the training dataset but theres one differnece, since Mnist has 70,000 images and we used 60,000 on the training dataset,
we will use the remaning 10,000 on the testing dataset '''




In [ ]:
# 3. Create DataLoaders

# DataLoader for training data
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)
'''in the parameter we added trained dataset that we made earlier, we grouped 32 samples together so instead of processing one image
at a time, we will process 32 together, it basically means that we are going to get a total batch of 60,000/32(i dont have a calculator...)
we will also randomizes the order of samples by shuffling at the beggining of each epoch to prevent the model from learning order-based
 patterns amd ensure the diversity of batches.
& to speed up Speeds up data preparation while the model is training, we used 2 parallel processes for data loading.

num_workers=2: Uses 2 parallel processes for data loading

'''

# DataLoader for test/validation data
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

''' Same with the train_model but the key difference is that we didn't shuffle the data to ensure consistent evaluation metrics &

make debugging/reproducibility easier'''


In [ ]:
# 4. Print shape of one batch

# Get the first batch from the training DataLoader
X_batch, y_batch = next(iter(train_loader))
'''we created an iterator to return the batchs from the train_loader, it will start from batch 1 and go through all the (60,000/32 batchs)'''

print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")

In [ ]:
# 5. Display sample images
''' Visualization of the first 6 images in a batch'''
images, labels = next(iter(train_loader))

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1) #plt.subplot(rows, columns, current position)
    plt.imshow(images[i].squeeze(), cmap='gray')
    '''we applied squeeze so we would remove the first dimension as im.show dont accept
   a certain number of Dimensions, we will go from [1, 28, 28] to [28, 28]'''

    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN4Layer, self).__init__()

        # First linear layer: input features -> hidden layer
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # Second linear layer: hidden layer -> hidden layer
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)

        # Third linear layer: hidden layer -> hidden layer
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)

        # Output layer: hidden layer -> number of classes (logits)
        self.layer4 = nn.Linear(hidden_dim, output_dim)

        # ReLU activation for non-linearity
        self.relu = nn.ReLU()

    # Defines how input data flows through the network
    def forward(self, X):
        # First hidden layer
        a1 = self.relu(self.layer1(X))

        # Second hidden layer
        a2 = self.relu(self.layer2(a1))

        # Third hidden layer
        a3 = self.relu(self.layer3(a2))

        # Output layer (raw scores / logits)
        output = self.layer4(a3)  # we said we'll use softmax, where is it? ¯\(ツ)/¯

        return output

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # Set the model to training mode
    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:
        # Move batch to the selected device
        X_batch = X_batch.to(device)         # shape: (batch_size, num_features)
        y_batch = y_batch.to(device)         # shape: (batch_size,)

        # Forward pass (outputs are logits)
        outputs = model(X_batch)             # shape: (batch_size, num_classes)
        loss = criterion(outputs, y_batch)

        # Backward pass & optimization
        optimizer.zero_grad()   # Clear previous gradients
        loss.backward()         # Compute gradients
        optimizer.step()        # Update model parameters

        running_loss += loss.item()

    # Average loss over all batches
    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
    # Set the model to evaluation mode
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # Move data to device
            X_batch = X_batch.to(device)     # shape: (batch_size, num_features)
            y_batch = y_batch.to(device)     # shape: (batch_size,)

            # Forward pass
            outputs = model(X_batch)         # shape: (batch_size, num_classes)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()

            # Apply Softmax to get probabilities
            probabilities = F.softmax(outputs, dim=1)

            # Pick the classes with highest probabilities
            predicted = torch.argmax(probabilities, dim=1)

            # Accuracy calculation
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    avg_loss = running_loss / len(test_loader)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
# Task 4: Define device, model, loss, optimizer:
''' we will start by creating a device object to manage where tensors/computations happen, if we got cuda it means GPU is availlable
otherwise its not'''
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model parameters
input_dim = 28 * 28   # MNIST images flattened and as they are 28*28 pixels
hidden_dim = 14      # Design choice, its the number of neurons in hidden layers
output_dim = 10       # Digits 0–9, each neuron in output layer represents one digit class

# Instantiate model,
''' we create instance of our neural network class & Moves entire model to specified device so all the layers' weights and biases

and any other parameters defined in the model'''
model = NN4Layer(input_dim, hidden_dim, output_dim).to(device)

# Print the model architecture
print("Model Architecture:\n")
print(model)
'''it should print Layer hierarchy, Input/output dimensions for each layer, Whether bias is included or not, and activation functions '''

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

In [ ]:
# Task 5: Start training for 20 epochs:

num_epochs = 20
learning_rate = 0.001

# Define criterion (loss function) - using CrossEntropyLoss as model now outputs logits
criterion = nn.CrossEntropyLoss()
# Define optimizer
optimizer = AdamW(model.parameters(), learning_rate)

In [ ]:
# Run Training
train_losses = []
val_losses = []
val_accuracies = []

print('Starting Training...')
for epoch in range(num_epochs):
    # Train one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

    # Validate
    val_loss, val_accuracy = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

print('Training Complete!')

#i know that the problem is with the sizes, ill return to it when im done with the rest of questions

In [ ]:
# Task 1: Write your code here:
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Set model to evaluation mode
model.eval()
# Get one batch from the test DataLoader
images, labels = next(iter(test_loader))
# Move images to device
images = images.to(device)
labels = labels.to(device)

with torch.no_grad():
    # Flatten images before passing to the model
    outputs = model(images.view(images.size(0), -1))
    predictions = torch.argmax(outputs, dim=1)

# Move tensors back to CPU for plotting
images = images.cpu()
labels = labels.cpu()
predictions = predictions.cpu()

# Plot first 6 predictions
plt.figure(figsize=(8, 4))
for i in range(6):
    plt.subplot(2, 3, i + 1)
    plt.imshow(images[i].squeeze(), cmap='gray')
    plt.title(f"True: {labels[i]} | Pred: {predictions[i]}")
    plt.axis('off')

plt.tight_layout()
plt.show()